# 19 - Continuous Actions In, Discrete DQN Actions Out

Train a DQN on [Gymnasium Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/).
The environment takes a **continuous vector of shape `(1,)`**, containing torque in `[-2, 2]`.
Only `[cos(theta), sin(theta)]` is observed. The model uses position and action history
to infer motion in this partially observable task. Episodes reset independently.

One `DiscretizedActionHead` produces **33 Q-values**. Each discrete category selects a complete row of a
fixed `head.action_vectors` table of shape `[33, 1]`. The table includes both limits and zero. The
environment receives `head.discrete_to_continuous(action_id)`; the transformer receives the **actual numeric
torque** in the next history row. The category ID is used only by the objective, never as a
transformer input. DQN chooses a category by epsilon-greedy selection over Q-values; it does not
need a separate policy head or a continuous actor.

```text
history of (previous torque, reward, current observation)
    -> transformer -> 33 Q-values -> one category ID
    -> head.discrete_to_continuous(ID) -> env.step(continuous vector)
    -> append the applied torque, reward, and next observation
```

This is an executable wiring and training example, not a trained swing-up agent or an in-context
adaptation benchmark. It uses a small transformer built from scratch, a local token vocabulary,
and CPU training: no model downloads, Box2D, or MuJoCo are needed. Install the repository's
`examples` extra first. Each cycle collects complete episodes into replay, then trains with DQN and separate delayed action and value heads. Replay is retained across cycles.

In [ ]:
from copy import deepcopy
import json
from pathlib import Path

import gymnasium as gym
from mouse_gym import EnvConfig, RolloutTracker, make_env
import numpy as np
from tokenizers import Tokenizer as TextTokenizer
from tokenizers.models import WordLevel
import torch
from torch import nn
from transformers import PreTrainedTokenizerFast

from mouse_core import AdamW, polyak_update
from mouse_core.data import (
    environment_episode_ranges,
    DataLoader,
    Datastore,
    EventLog,
    Tokenizer,
    save_tokenizer,
    to_device,
)
from mouse_core.models import (
    Model,
    DecodeCache,
    prepare_decode,
    Backbone,
    Embedding,
    DiscretizedActionHead,
    load_model,
    save_model,
)
from mouse_core.objectives import DqnObjective, boundary_discount

# Artifacts
SAVE_ARTIFACTS = False
OUTPUT_DIR = Path("runs/pendulum-discrete")

# Environment
MAX_ACTIONS = 33
episode_limit = gym.spec("Pendulum-v1").max_episode_steps
assert episode_limit is not None
MAX_STEPS_PER_EPISODE = episode_limit  # native limit

# Model
HIDDEN_DIM = 32
BACKBONE_LAYERS = 1
BACKBONE_HEADS = 2
NUM_FREQUENCIES = 4
HEAD_LAYERS = 1
HEAD_SCALE = 0.1  # final prediction linear's initialization

# Collection and rollout
EPISODES_PER_CYCLE = 2
EPSILON = 0.2

# Training
NUM_CYCLES = 2
TRAIN_STEPS = 2

# Optimizer learning rates
LR_EMBEDDING = 3e-4  # token embeddings and numeric Fourier features
LR_BACKBONE = 3e-4  # backbone
LR_HEADS = 3e-4  # prediction heads and their input RMSNorms

# Target interpolation rates
POLYAK_TAU_EMBEDDING = 0.01  # delayed embedding
POLYAK_TAU_BACKBONE = 0.01  # delayed backbone
POLYAK_TAU_VALUE = 0.01  # delayed value head
POLYAK_TAU_ACTION = 1.0  # independent action-selection head

# Random seeds
SEED = 7

# Runtime
device = torch.device("cpu")
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.manual_seed(SEED)
action_rng = np.random.default_rng(SEED)


## Build Environment

The observation wrapper exposes only cosine and sine before outputs reach the event log.
The environment keeps its continuous `Box` action space. The conversion happens in the collector,
so the model's discrete output and the physical action are both visible.
`head.action_vectors[k]` is a complete action vector, even though Pendulum has just one component.
For a multidimensional environment the same lookup would use a `[K, action_dim]` table, still with
one categorical output over its `K` rows. A Cartesian grid would have the product of the per-axis
bin counts; an explicitly chosen finite list of vectors need not contain every combination.

The table and its ordering are part of the policy. Changing either after training changes what
its outputs mean. The head owns this table as a fixed model buffer. It moves with the model and is saved in the
model checkpoint automatically. No separate action-table file is needed.

In [ ]:
def make_pendulum():
    pendulum = gym.make("Pendulum-v1")
    bounds = pendulum.observation_space
    assert isinstance(bounds, gym.spaces.Box)
    return gym.wrappers.TransformObservation(
        pendulum,
        func=lambda observation: observation[:2],
        observation_space=gym.spaces.Box(
            low=bounds.low[:2],
            high=bounds.high[:2],
            dtype=np.float32,
        ),
    )


env = make_env(EnvConfig(
    env_fn=make_pendulum,
    name="Pendulum-v1",
    seed=SEED,
    seed_mode="stream",
))
assert env.observation_space.shape == (2,)
assert isinstance(env.action_space, gym.spaces.Box)
assert env.action_space.shape == (1,)
action_vectors = torch.linspace(
    float(env.action_space.low[0]), float(env.action_space.high[0]), MAX_ACTIONS,
)[:, None]
assert action_vectors.shape == (MAX_ACTIONS, 1)
assert all(env.action_space.contains(vector.numpy()) for vector in action_vectors)
assert action_vectors[MAX_ACTIONS // 2, 0] == 0.0


## Data Pipeline

Each row contains the transition **into** its observation:

| Row field | Transformer input | DQN objective |
| --- | --- | --- |
| `torque` | Applied continuous torque, as a numeric token | No |
| `cos_theta`, `sin_theta` | Separate numeric tokens for cosine and sine | No |
| `reward` | Numeric token | Reward |
| `episode_start` | `reset` token on the initial observation | No |
| `action` | **No** | Discrete category ID |
| `terminated`, `truncated` | No | Boundary discounts |
| `episode_index` | No | Groups the history and loss by episode |

Pendulum's vector has one component, so the collector stores that physical value as scalar
`torque`. It never replaces it with the category ID. A reset frame has no torque or action ID because no action was applied. DQN uses row `t+1`'s ID and reward with row `t`'s prediction; nullable actions produce an execution mask during packing.

The small text vocabulary supplies only **field identities**, such as `torque` and `cos`.
Actual floating-point values travel in numeric tensors. Existing Fourier features turn these
values into additions to the field embeddings. The Fourier windows below are wider than the
physical ranges so opposite physical limits do not coincide at the periodic encoding endpoints.

In [ ]:
def on_episode_start(ctx):
    return ctx.get("episode_start", False)

vocabulary = {
    "[UNK]": 0, "torque": 1, "cos": 2, "sin": 3,
    "reward": 4, "reset": 5, "readout": 6,
}
text_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=TextTokenizer(WordLevel(vocab=vocabulary, unk_token="[UNK]")),
    unk_token="[UNK]",
)

tokenizer = Tokenizer(
    step_field="step_index",
    tokenizer=text_tokenizer,
    input_fields=[
        {
            'type': 'token',
            'provider': 'coordinator',
            'format': 'reset',
            'when': on_episode_start,
        },
        {
            'type': 'numeric',
            'provider': None,
            'input_field': 'torque',
            'format': 'torque',
            'fourier_min': -4.0,
            'fourier_max': 4.0,
        },
        {
            'type': 'numeric',
            'provider': None,
            'input_field': 'cos_theta',
            'format': 'cos',
            'fourier_min': -2.0,
            'fourier_max': 2.0,
        },
        {
            'type': 'numeric',
            'provider': None,
            'input_field': 'sin_theta',
            'format': 'sin',
            'fourier_min': -2.0,
            'fourier_max': 2.0,
        },
        {
            'type': 'numeric',
            'provider': None,
            'input_field': 'reward',
            'format': 'reward',
            'fourier_min': -32.0,
            'fourier_max': 32.0,
        },
        {
            'type': 'token',
            'provider': None,
            'input_field': 'action_requested',
            'format': '\n',
            'head_output': True,
        },
    ],
    objective_fields=[
        {"input_field": "action",
        "missing_value": None,
    },
        {"input_field": "reward",
        "missing_value": 0.0,
    },
        {"input_field": "terminated",
        "missing_value": False,
    },
        {"input_field": "truncated",
        "missing_value": False,
    },
    ],
)


## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference
uses the same encoder and returns cache and readout-mask metadata. A `ModuleDict`
called `modules` holds references to the online components for checkpointing and
mode changes; it is not called for predictions.

The delayed encoder is a frozen copy. Separate backup value/action heads track
the online Q head(s) with independent Polyak rates. Only online parameters enter
the optimizer.

The Q head owns the fixed action-vector table and its continuous/discrete
conversion methods. The action table is saved as part of the Q head.



In [ ]:
class Encoder(Model):
    def __init__(self, *, hidden_dim, vocab_size, num_frequencies, num_layers, num_heads,
                 train_kernel, decode_kernel, dtype):
        super().__init__()
        self.embedding = Embedding(hidden_dim=hidden_dim, vocab_size=vocab_size,
                                   num_frequencies=num_frequencies)
        self.backbone = Backbone(
            architecture="qwen3", hidden_dim=hidden_dim, num_layers=num_layers,
            num_heads=num_heads, train_kernel=train_kernel,
            decode_kernel=decode_kernel, dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to_tensors(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_ids"]
        head_step_ids = tensors["head_step_ids"]
        embed = self.embedding(
            token_ids, numeric_values=numeric_values, numeric_mask=numeric_mask,
        )
        head_valid_mask = None
        if use_cache:
            embed, lengths, head_token_ids, head_valid_mask = prepare_decode(
                embeds=embed, tokens=tokens, sequence_ids=sequence_ids,
                head_token_ids=head_token_ids, head_step_ids=head_step_ids,
            )
            if cache is None:
                cache = DecodeCache(sessions=(self.backbone.decode_session(batch_size=tokens.B),))
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape, features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "last_hidden_state": features,
            "head_token_ids": head_token_ids,
            "head_valid_mask": head_valid_mask,
            "cache": cache,
        }

class QHead(DiscretizedActionHead):
    def __init__(self, *, hidden_dim, action_vectors, num_layers, scale):
        super().__init__(
            in_features=hidden_dim, action_vectors=action_vectors,
            hidden_dim=hidden_dim, num_layers=num_layers, scale=scale,
            propagate_gradient=1.0,
        )

encoder = Encoder(
    hidden_dim=HIDDEN_DIM, vocab_size=len(text_tokenizer), num_frequencies=NUM_FREQUENCIES,
    num_layers=BACKBONE_LAYERS, num_heads=BACKBONE_HEADS,
    train_kernel="reference", decode_kernel="flex", dtype=torch.float32,
).train().to(device)

q_head = QHead(
    hidden_dim=encoder.backbone.hidden_dim, action_vectors=action_vectors,
    num_layers=HEAD_LAYERS, scale=HEAD_SCALE,
).train().to(device)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict({"encoder": encoder, "q_head": q_head})

delayed_encoder = deepcopy(encoder).requires_grad_(False).eval()
backup_value_head = deepcopy(q_head).requires_grad_(False).eval()
backup_action_head = deepcopy(q_head).requires_grad_(False).eval()

head = q_head
print(encoder)
print(q_head)
print(f"{sum(p.numel() for p in modules.parameters()):,} parameters; {MAX_ACTIONS} Q-values")
for action_id in (0, MAX_ACTIONS // 2, MAX_ACTIONS - 1):
    vector = head.discrete_to_continuous(action_id)
    recovered_id = int(head.continuous_to_discrete(vector).item())
    assert recovered_id == action_id
    print(f"category {action_id:2d} -> continuous vector {vector.tolist()} -> category {recovered_id}")

query = torch.tensor([0.31])
nearest_id = head.continuous_to_discrete(query)
print(
    f'Off-grid {query.tolist()} -> category {nearest_id.item()} -> {head.discrete_to_continuous(nearest_id).tolist()}',
)


## Collect Rollouts

Mouse-gym's first step and the step after a boundary are reset frames: their passed-in action is
ignored. We request those frames explicitly, then select actual actions only after observing a
state. Epsilon exploration samples a **category ID**, so all executed actions belong to the table.
Each action is decoded once; the same torque sent to the environment is stored in history.

At the final observation, stop collecting that episode. Keep its final reward and boundary flags.
The next episode starts a separate attention/loss group. Replay retains earlier cycles' episodes.
`RolloutTracker` tracks each completed output and its episode and step counters.
The coordinator marks episode starts; the agent records the selected torque, and the environment records `cos_theta` and `sin_theta`. The tracker controls the loop. The raw event log supplies the history
row from those records; replay stores the original events.


In [ ]:
event_log = EventLog()
tracker = RolloutTracker()

@torch.no_grad()
def collect_episode() -> tuple[float, list[dict]]:
    modules.eval()
    event_start = len(event_log)
    episode_start = tracker.output is None or tracker.ended
    episode_index = tracker.next_episode_index
    if episode_start:
        coordinator_event = event_log.event(provider='coordinator', episode_index=episode_index, step_index=(0 if episode_start else tracker.step_index + 1), payload={
            "episode_start": True,
        })
        store.append(data=coordinator_event)
    output = env.step(None)
    tracker.update(output=output)

    environment_event = event_log.event(provider='environment', episode_index=tracker.episode_index, step_index=tracker.step_index, payload={
        'cos_theta': tracker.observation[0],
        'sin_theta': tracker.observation[1],
        **({"reward": tracker.reward} if tracker.reward != 0 else {}),
        **({"terminated": True} if tracker.terminated else {}),
        **({"truncated": True} if tracker.truncated else {}),
        **({"action_requested": True} if tracker.action_requested else {}),
    })
    store.append(data=environment_event)
    history = event_log[event_start:]
    while not tracker.ended:
        event_start = len(event_log)
        episode_start = tracker.output is None or tracker.ended
        episode_index = tracker.next_episode_index
        if episode_start:
            coordinator_event = event_log.event(provider='coordinator', episode_index=episode_index, step_index=(0 if episode_start else tracker.step_index + 1), payload={
                "episode_start": True,
            })
            store.append(data=coordinator_event)
        inputs = tokenizer.pack_rows(rows=[history], continuing=None)
        out = encoder(inputs)
        q_values = q_head(out["z"])[-1]
        if action_rng.random() < EPSILON:
            action_id = int(action_rng.integers(MAX_ACTIONS))
        else:
            action_id = int(q_values.argmax().item())
        continuous_action = head.discrete_to_continuous(action_id).cpu().numpy()
        assert env.action_space.contains(continuous_action)
        agent_event = event_log.event(provider='agent', episode_index=episode_index, step_index=tracker.step_index + 1, payload={
            'action': action_id,
            'torque': float(continuous_action[0]),
        })
        store.append(data=agent_event)
        output = env.step(continuous_action)
        tracker.update(output=output)

        environment_event = event_log.event(provider='environment', episode_index=tracker.episode_index, step_index=tracker.step_index, payload={
            'cos_theta': tracker.observation[0],
            'sin_theta': tracker.observation[1],
            **({"reward": tracker.reward} if tracker.reward != 0 else {}),
            **({"terminated": True} if tracker.terminated else {}),
            **({"truncated": True} if tracker.truncated else {}),
            **({"action_requested": True} if tracker.action_requested else {}),
        })
        store.append(data=environment_event)
        history.extend(event_log[event_start:])
    return tracker.episodes[-1].reward, history


## Train

The agent event's `action` is the objective's integer category ID. All 33 categories represent
valid vectors, so no action mask is needed. The delayed action head chooses the next category
and the delayed value head evaluates it. No continuous-action maximization or new loss is necessary.

This example learns the discounted return within each complete 200-step Pendulum episode.
The native time limit ends that finite horizon: `gamma_truncated=0` stops the return after
its final reward, while the recorded environment flag remains `truncated=True`.
`gamma_terminated=0` also stops the return at a true terminal. `cross_group_backups="fault"`
raises if a sampled episode is cut short and a target needs data beyond its group.
Replay samples complete episodes; reset observations start separate attention/loss groups.

In [ ]:
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=0.99, gamma_terminated=0.0, gamma_truncated=0.0,
    ),
    temperature=0.0,
    cross_group_backups="fault",
    gate=None,
)
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(lr={'embedding': LR_EMBEDDING, 'backbone': LR_BACKBONE}),
        {'params': q_head.parameters(), 'lr': LR_HEADS},
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
)

def run_train(*, cycle: int, num_steps: int) -> tuple[float, dict[str, float | torch.Tensor]]:
    if num_steps < 1:
        raise ValueError("num_steps must be positive")
    loader = DataLoader(
        stores=[store],
        events_budget=(3 * MAX_STEPS_PER_EPISODE + 2) * EPISODES_PER_CYCLE,
        batch_size=1,
        sample_ranges=environment_episode_ranges,
        transform=tokenizer.encode,
        num_workers=0,
        seed=SEED + cycle,
    )
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    try:
        modules.train()
        for _ in range(num_steps):
            inputs, objective_data, group_id = loader.next_batch()
            out = encoder(inputs)
            with torch.no_grad():
                delayed_out = delayed_encoder(inputs)
            losses, metrics = objective(
                objective_data=to_device(data=objective_data, device=device),
                group_id=group_id.to(device),
                predictions=q_head(out['z']),
                backup_value_predictions=backup_value_head(delayed_out['z']),
                backup_action_predictions=backup_action_head(delayed_out['z']),
            )
            loss = torch.stack(tuple(losses.values())).sum()
            assert torch.isfinite(loss)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modules.parameters(), float("inf"), error_if_nonfinite=True)
            optimizer.step()
            polyak_update(
                target=delayed_encoder.embedding,
                source=encoder.embedding,
                tau=POLYAK_TAU_EMBEDDING,
            )
            polyak_update(target=delayed_encoder.backbone, source=encoder.backbone, tau=POLYAK_TAU_BACKBONE)
            polyak_update(target=backup_value_head, source=q_head, tau=POLYAK_TAU_VALUE)
            polyak_update(target=backup_action_head, source=q_head, tau=POLYAK_TAU_ACTION)
        assert loss is not None
        loss_value = float(loss.detach())
        return (loss_value, metrics)
    finally:
        loader.close()


## Run

Each cycle collects complete episodes, then runs `TRAIN_STEPS` optimizer updates.


In [ ]:
store = Datastore(name="pendulum-replay")
training_log = []
last_history: list[dict] | None = None
try:
    for cycle in range(NUM_CYCLES):
        returns = []
        for episode in range(EPISODES_PER_CYCLE):
            episode_return, last_history = collect_episode()
            returns.append(episode_return)
        loss, metrics = run_train(cycle=cycle, num_steps=TRAIN_STEPS)
        training_log.append({'cycle': cycle, 'mean_return': float(np.mean(returns)), 'loss': loss})
        print(
            f"cycle={cycle} | mean return={np.mean(returns):.1f} | loss={loss:.4f} | mean Q={float(metrics['q_values_mean']):.3f} | replay rows={len(store)}",
        )
finally:
    env.close()

assert last_history is not None, "Collect at least one episode before inspecting its history"
print(f"Finished {NUM_CYCLES * EPISODES_PER_CYCLE} episodes and {NUM_CYCLES * TRAIN_STEPS} updates.")


## Inspect the two representations

Changing only `action` must not change transformer input tokens or numeric values. Changing
torque must change the numeric input even when the category ID is held fixed. This check also
confirms that the recorded physical action agrees with the lookup table.


In [ ]:
example_event = next(event for event in last_history if event["provider"] == "agent")
example = example_event["payload"]
assert example["torque"] == float(head.discrete_to_continuous(example["action"])[0])
assert head.continuous_to_discrete(torch.tensor([example["torque"]])).item() == example["action"]
all_ids = torch.arange(head.out_features, device=device)
assert torch.equal(head.continuous_to_discrete(head.discrete_to_continuous(all_ids)), all_ids)
encoded = tokenizer.encode([example_event])[0]
id_changed = tokenizer.encode([{**example_event, "payload": {**example, "action": (example["action"] + 1) % MAX_ACTIONS}}])[0]
np.testing.assert_array_equal(encoded.token_ids, id_changed.token_ids)
np.testing.assert_array_equal(encoded.numeric_values, id_changed.numeric_values)
assert encoded.objective_fields["action"] != id_changed.objective_fields["action"]

low = tokenizer({**example_event, "payload": {**example, "torque": -2.0}})
high = tokenizer({**example_event, "payload": {**example, "torque": 2.0}})
np.testing.assert_array_equal(low["token_ids"], high["token_ids"])
assert low["numeric_values"][0] == -0.5 and high["numeric_values"][0] == 0.5
assert encoder.embedding.fourier is not None
assert encoder.embedding.fourier.proj.weight.detach().abs().sum().item() > 0.0
print({"category_for_loss": example["action"], "torque_for_transformer": example["torque"]})
print("Confirmed: continuous numeric input, discrete objective target.")


## Save Locally

`save_model` includes the head's table as a persistent buffer. `load_model` restores the special
head, Q-values, row ordering, and both conversions. Set `SAVE_ARTIFACTS=True` to save a local bundle
and check the table after reloading. The environment settings remain separate from the model.
The local text tokenizer and the mouse-core packing specification are saved separately because
both use a file named `tokenizer.json`.


In [ ]:
if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_model(model=modules, path=OUTPUT_DIR / "model")
    text_tokenizer.save_pretrained(OUTPUT_DIR / "text_tokenizer")
    save_tokenizer(tokenizer=tokenizer, path=OUTPUT_DIR / "packing")
    (OUTPUT_DIR / "environment.json").write_text(json.dumps({
        "id": "Pendulum-v1",
        "max_episode_steps": MAX_STEPS_PER_EPISODE,
        "observation_columns": ["cos_theta", "sin_theta"],
        "observation_indices": [0, 1],
        "action_shape": [head.action_dim],
        "action_dtype": "float32",
        "action_columns": ["torque"],
    }, indent=2) + "\n")
    loaded = load_model(
        model=deepcopy(modules),
        repo_id_or_path=str(OUTPUT_DIR / "model"),
    )
    restored_head = loaded["q_head"]
    assert isinstance(restored_head, DiscretizedActionHead)
    assert torch.equal(restored_head.action_vectors, head.action_vectors.cpu())
    assert torch.equal(restored_head.continuous_to_discrete(restored_head.action_vectors), all_ids.cpu())
    print(
        f'Saved and reloaded model with action table, plus tokenizers and environment settings: {OUTPUT_DIR.resolve()}',
    )


## What extra machinery did this require?

1. **A `DiscretizedActionHead`:** one Q-value per fixed table row, exact ID-to-vector lookup,
   and nearest-vector quantization in the other direction. The table is a non-trainable buffer
   included in model copies, device moves, and checkpoints.
2. **Both action representations in collection:** the ID goes to DQN's gather operation; the actual
   continuous components go into the next transformer history row. Keep the table ordering fixed
   during collection, optimization, and inference.
3. **Numeric action input fields:** reuse `Tokenizer(type="numeric")` and the embedding's
   `num_frequencies`, just as for continuous observations. No new embedding API is required.

Replay, the delayed agent, DQN, and Polyak updates are existing components; the loss is
unchanged. Arbitrary offline continuous actions can use `continuous_to_discrete`, but their
quantization error remains an approximation in the training targets. For multidimensional actions,
Euclidean distance uses the table's physical units, so coordinate scaling matters. A Cartesian grid
can become large, and housing the table in the head does not make nearby actions share Q-values.
